# StockSense — Predicting Daily Stock Movement with Machine Learning

**Goal:** Build a complete ML pipeline that predicts whether a stock will close *higher or lower* the next trading day.

**Why this problem?**
I chose this problem because stock price movement is one of the most studied, most difficult prediction tasks in ML. It's noisy, non-stationary, and full of many, common pitfalls with data. This makes it an excellent project for learning: every concept I studied here — data leakage, feature engineering, class imbalance, overfitting — is something engineers genuinely wrestle with in production systems.

**End deliverable:** A single, fully documented Jupyter notebook — raw data in, trained models and evaluation charts out.

## Imports

All dependencies are declared at the top of the notebook in a single cell. This is standard practice in data science notebooks. Full requirements can be found in the requirments.txt file in the project root directory.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as npy
import matplotlib.pyplot as plt
import seaborn as sbn

## Acquiring the Data

**What I'm doing:** Downloading 5 years of daily OHLCV data for Apple Inc. (AAPL) from Yahoo Finance via the `yfinance` API.

**Why Apple (AAPL)?**
- Highly liquid — prices reflect genuine market activity
- Has 5 years of history that gives ~1,260 trading days, which is enough for meaningful ML without being unwieldy
- No survivorship bias concern for a single ticker used as a learning exercise
- Free and accessible via `yfinance`

**OHLCV defined:**
| Column | Meaning |
|--------|---------|
| Open | Price at market open |
| High | Intraday highest price |
| Low | Intraday lowest price |
| Close | Price at market close |
| Volume | Number of shares traded |
| Adj Close | Close adjusted for dividends and stock splits |

> 🔑 Data Clarification: I'm using raw Close rather than Adj Close for simplicity. For a production system analysing longer time horizons or dividend-heavy stocks, Adj Close would be the best choice, as it removes price discontinuities caused by dividends and stock splits.

In [ ]:
df = yf.download('AAPL', period='5y')

### Inspecting the data

Before any transformation, I run three standard inspection calls, called a **sanity check**. You will waste hours debugging a model that has a broken input.

- `df.head()` — confirm the column names, index type (should be `DatetimeIndex`), and that values look reasonable
- `df.info()` — check dtypes and the count of non-null entries per column. Any column with missing values will show a lower count than the total rows
- `df.describe()` — check the statistical summary: min, max, mean, std. A negative minimum Volume, for example, would immediately flag a data issue

> 🔑 **What I'm watching for:**
> - Missing values (`NaN`s) — yfinance data is generally clean, but market holidays and early data can introduce gaps
> - Outliers in data or corrupted data is very important to confirm before doing features and splitting data.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Splitting the data — and why it happens before anything else

I'm dividing the dataset into three separate, non-overlapping sets before engineering any features or building any models. The split has to happen here. Doing it later risks contaminating the training data with information from the future.

| Set            | Role                                     | Approximate size               |
|----------------|------------------------------------------|--------------------------------|
| **Train**      | The model learns from this               | ~70% — first ~880 trading days |
| **Validation** | Used to compare and tune models          | ~15% — next ~190 days          |
| **Test**       | Final evaluation, looked at exactly once | ~15% — last ~190 days          |

**Why split chronologically and not randomly?**

Stock data is sequential — each row follows the previous one in real time. Randomly shuffling the rows before splitting, which is perfectly valid for most ML datasets, would mix future dates into the training set and past dates into the test set. The model would have effectively seen the future during training, making its test performance look far better than it really is. This is called **data leakage** — one of the most common ways ML systems appear to work in development and then fail immediately in production.

The split here is a clean date-ordered slice. No shuffling.

**Why keep a validation set separate from the test set?**

Every time I look at test set performance and adjust the model based on what I see, that test set is no longer a fair measure of unseen data — I've been optimizing toward it without realizing it. The validation set exists to absorb all tuning decisions. The test set stays untouched until the very end, and whatever it returns is the final reported result with no further changes.

In [ ]:
data_size = len(df)
# 70% training data, 15% validation data, 15% test data put into new data frames
train_df = df.iloc[:int(data_size * 0.7)]
val_df = df.iloc[int(data_size * 0.7):int(data_size * 0.85)]
test_df = df.iloc[int(data_size * 0.85):]

In [ ]:
# Ensure data was split correctly
print(f'Train set shape: {train_df.shape}')
print(f'Val set shape:   {val_df.shape}')
print(f'Test set shape:  {test_df.shape}')
if train_df.shape[0] + val_df.shape[0] + test_df.shape[0] == data_size:
    print('Data split correctly!')
else:
    print('Data was not split correctly!')

## Feature Engineering

Raw closing prices on their own give a model almost nothing to work with. The fact that AAPL closed at $182 today doesn't tell a model whether prices have been rising or falling, whether buyers have been more aggressive than sellers, or how much the stock has been swinging around. That context has to be computed and added to the dataset as new columns.

This is called **feature engineering** — turning raw data into informative inputs. It tends to be where the most consequential decisions in an ML pipeline live. A model can only learn from patterns that are encoded somewhere in its inputs; if the signal isn't there, no amount of complexity can pull it out of thin air.

I'm computing six features. Each one captures something different about the stock's behavior.

### Simple Moving Average — SMA10 and SMA50

The SMA is the average closing price over the last N days, computed by sliding that window forward one day at a time.

```python
df['SMA10'] = df['Close'].rolling(window=short_window).mean()
df['SMA50'] = df['Close'].rolling(window=long_window).mean()
```

The 10-day version responds quickly to price changes — it tracks short-term momentum. The 50-day is slower and smoother, representing the longer-term trend underneath. Neither is particularly useful on its own; the relationship between them is what matters.

When the 10-day average crosses *above* the 50-day, it signals that recent prices are outpacing the longer trend — a bullish indicator traders call the "golden cross." When it crosses below, the reverse: bearish momentum, known as the "death cross." These crossover patterns are some of the most watched signals in technical analysis, and giving the model visibility into both moving averages lets it learn whether those patterns have actual predictive power on this data.

In [ ]:
short_window = 10
long_window = 50
# Perform a Simple Moving Average of closing over last n days
# short term avgs > long term avgs = buy signal
df['SMA10'] = df['Close'].rolling(window=short_window).mean()
df['SMA50'] = df['Close'].rolling(window=long_window).mean()
df[['Close', 'SMA10', 'SMA50']].head(50)

### Exponential Moving Average — EMA12 and EMA26

The EMA works like the SMA but weights recent prices more heavily. A price from yesterday influences the average more than a price from two weeks ago. This makes EMAs react faster to new information — useful for detecting shifts in momentum before the SMA picks them up.

```python
df['EMA12'] = df['Close'].ewm(span=fast_decay).mean()
df['EMA26'] = df['Close'].ewm(span=slow_decay).mean()
```

I'm using spans of 12 and 26 specifically because they're the standard inputs to MACD.

### MACD

```python
df['MACD'] = df['EMA12'] - df['EMA26']
```

MACD is simply the difference between the two EMAs. When positive, the short-term average is running ahead of the longer-term trend — bullish momentum. When negative, the short-term average is lagging — bearish momentum. It's a compact way of encoding both the direction and the strength of a trend into one number.

In [ ]:
fast_decay = 12
slow_decay = 26
# Performs a Exponential Moving Average
# Similar to SMA but gives more weight to recent prices
df['EMA12'] = df['Close'].ewm(span=fast_decay).mean()
df['EMA26'] = df['Close'].ewm(span=slow_decay).mean()
# EMA_12 > EMA_26 => Bullish; EMA_12 < EMA_26 => Bearish
df['MACD'] = df['EMA12'] - df['EMA26']
df[['Close', 'EMA12', 'EMA26', 'MACD']].head(50)

### RSI — Relative Strength Index

RSI measures whether a stock has been bought or sold too aggressively relative to its own recent history. Where the moving averages track price *direction*, RSI tracks market *sentiment* — whether buyers or sellers have been in control. These are fundamentally different pieces of information, which is why RSI is worth including alongside the moving average features.

>The formula compares the average size of daily price gains to the average size of daily losses over a 14-day window:
> - RSI = 100 − (100 / (1 + avg_gain / avg_loss))

The result is always between 0 and 100:
- **Above 70** — overbought. The stock has been rising aggressively and may be due for a pullback
- **Below 30** — oversold. The stock has been falling hard and may be due for a bounce
- **Around 50** — neither; relatively neutral

I'm computing it manually rather than using a library like *ta*.

In [ ]:
# Finding daily change in price
daily_change = df['Close'].diff()
rsi_window = 14
# Separating daily_change col into two separate col's: gain/loss
gain = daily_change.where(daily_change > 0, 0)
loss = -daily_change.where(daily_change < 0, 0)
# Compute rolling avg's
avg_gain = gain.rolling(window=rsi_window).mean()
avg_loss = loss.rolling(window=rsi_window).mean()
# Compute RS
rs = avg_gain / avg_loss
# Compute RSI so values are 0 -> 100 and not 0 -> inf
df['RSI'] = 100 - (100 / (1 + rs))

In [ ]:
# Verify 0 <= x <= 100
df['RSI'].describe()

**Observation:** The RSI series oscillates between roughly 20 and 80 for most of the period, with occasional spikes toward the extremes during sharp market moves. Values consistently staying above 70 or below 30 for extended periods would be a sign the indicator is behaving unexpectedly — seeing it spend most of its time in the middle range confirms the calculation is working correctly.

In [ ]:
# Visualization: verifying values are generally between 20-80
df['RSI'].plot()